In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/spotify_clean.csv')
print(f"✅ Loaded: {df.shape}")

SIMILARITY_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'mood_score', 'dance_energy'
]

# Encode genre as a weighted feature
le = LabelEncoder()
df['genre_encoded'] = le.fit_transform(df['track_genre'])

# Scale audio features
scaler = StandardScaler()
audio_scaled = scaler.fit_transform(df[SIMILARITY_FEATURES])

# Genre gets 3x weight — strong genre preference
genre_scaled = df['genre_encoded'].values.reshape(-1, 1) * 3

# Popularity gets 2x weight — prefer similar popularity level
pop_scaled = (df['popularity'].values.reshape(-1, 1) / 100) * 2

# Combine into final feature matrix
feature_matrix = np.hstack([audio_scaled, genre_scaled, pop_scaled])

print(f"✅ Feature matrix with genre + popularity weighting: {feature_matrix.shape}")
print(f"   Audio features: {len(SIMILARITY_FEATURES)}")
print(f"   + Genre weight (3x)")
print(f"   + Popularity weight (2x)")

✅ Loaded: (113549, 27)
✅ Feature matrix with genre + popularity weighting: (113549, 13)
   Audio features: 11
   + Genre weight (3x)
   + Popularity weight (2x)


In [9]:
def find_similar_tracks(track_name, n=5, artist=None):
    """Find tracks most similar to a given track using cosine similarity."""
    
    mask = df['track_name'].str.lower().str.contains(track_name.lower())
    
    if artist:
        mask = mask & df['artists'].str.lower().str.contains(artist.lower())
    
    matches = df[mask]
    
    if len(matches) == 0:
        return None, f"Track '{track_name}' not found"
    
    # Pick most popular version if multiple exist
    idx = matches['popularity'].idxmax()
    track_vector = feature_matrix[idx].reshape(1, -1)
    
    similarities = cosine_similarity(track_vector, feature_matrix)[0]
    similar_indices = similarities.argsort()[::-1][1:n+1]
    
    results = df.iloc[similar_indices][
        ['track_name','artists','track_genre','popularity','danceability','energy','valence']
    ].copy()
    results['similarity_score'] = similarities[similar_indices].round(3)
    
    source_track = df.iloc[idx]
    return source_track, results

# Test — now picks most popular version
source, similar = find_similar_tracks("Blinding Lights")
if source is not None:
    print(f"Source track: {source['track_name']} by {source['artists']}")
    print(f"Genre: {source['track_genre']} | Popularity: {source['popularity']}")
    print(f"\nTop 5 similar tracks:")
    print(similar[['track_name','artists','track_genre','popularity','similarity_score']].to_string(index=False))

Source track: Blinding Lights by The Weeknd
Genre: pop | Popularity: 91

Top 5 similar tracks:
                                           track_name                             artists track_genre  popularity  similarity_score
                                          Unstoppable                                 Sia         pop          81               1.0
Playground (from the series Arcane League of Legends) Bea Miller;Arcane;League of Legends   indie-pop          70               1.0
                                        Courtesy Call                Thousand Foot Krutch       metal          72               1.0
                                               感谢你曾来过                            阿涵;Ayo97    mandopop          61               1.0
                                               愛上你算我賤                         NICKTHEREAL    mandopop          62               1.0


In [3]:
test_tracks = [
    "Shape of You",
    "Bohemian Rhapsody", 
    "Bad Guy",
    "Despacito",
]

for track in test_tracks:
    source, similar = find_similar_tracks(track)
    if source is not None:
        print(f"\n🎵 '{source['track_name']}' by {source['artists']}")
        print(f"   Genre: {source['track_genre']} | Popularity: {source['popularity']}")
        print(f"   Similar tracks:")
        for _, row in similar.head(3).iterrows():
            print(f"   → {row['track_name']} by {row['artists']} (similarity: {row['similarity_score']})")
    else:
        print(f"\n❌ '{track}' not found")


🎵 'Shape Of You' by Andrew Foy
   Genre: guitar | Popularity: 24
   Similar tracks:
   → Always Will by Steve Martin;Steep Canyon Rangers (similarity: 0.981)
   → Minuet in F, K.1d by Wolfgang Amadeus Mozart;Erik Smith (similarity: 0.98)
   → Brandenburg Concerto No. 3 in G, BWV 1048: 1. (Allegro) by Johann Sebastian Bach;Berliner Philharmoniker;Herbert von Karajan (similarity: 0.98)

🎵 'Bohemian Rhapsody' by Hayseed Dixie
   Genre: bluegrass | Popularity: 22
   Similar tracks:
   → All Around Me by Justin Bieber (similarity: 0.977)
   → Milonguita (Esthercita) by Alfredo De Angelis;Dante Carlos;Orquesta de Alfredo De Angelis (similarity: 0.976)
   → Prohibido by Alfredo De Angelis;Oscar Larroca;Carlos Dante (similarity: 0.975)

🎵 'bad guy' by Twinkle Twinkle Little Rock Star
   Genre: children | Popularity: 28
   Similar tracks:
   → Don't Save Me by Twinkle Twinkle Little Rock Star (similarity: 0.997)
   → everything i wanted by Twinkle Twinkle Little Rock Star (similarity: 0.995)
 

In [4]:
def get_genre_mood_profile(genre):
    """Get the audio fingerprint of a genre."""
    genre_df = df[df['track_genre'].str.lower() == genre.lower()]
    
    if len(genre_df) == 0:
        # Partial match
        genre_df = df[df['track_genre'].str.lower().str.contains(genre.lower())]
    
    if len(genre_df) == 0:
        return None
    
    profile = {
        'genre': genre_df['track_genre'].iloc[0],
        'track_count': len(genre_df),
        'avg_popularity': round(genre_df['popularity'].mean(), 1),
        'avg_danceability': round(genre_df['danceability'].mean(), 3),
        'avg_energy': round(genre_df['energy'].mean(), 3),
        'avg_valence': round(genre_df['valence'].mean(), 3),
        'avg_tempo': round(genre_df['tempo'].mean(), 1),
        'avg_acousticness': round(genre_df['acousticness'].mean(), 3),
        'mood_score': round(genre_df['mood_score'].mean(), 3),
        'top_artists': genre_df.nlargest(5, 'popularity')['artists'].tolist()
    }
    
    # Mood label
    if profile['avg_valence'] > 0.6 and profile['avg_energy'] > 0.6:
        profile['mood'] = 'Energetic & Happy'
    elif profile['avg_valence'] > 0.6 and profile['avg_energy'] <= 0.6:
        profile['mood'] = 'Calm & Positive'
    elif profile['avg_valence'] <= 0.4 and profile['avg_energy'] > 0.6:
        profile['mood'] = 'Intense & Dark'
    elif profile['avg_valence'] <= 0.4 and profile['avg_energy'] <= 0.4:
        profile['mood'] = 'Melancholic & Quiet'
    else:
        profile['mood'] = 'Balanced'
    
    return profile

# Test
for genre in ['pop', 'jazz', 'metal', 'k-pop', 'sad']:
    profile = get_genre_mood_profile(genre)
    if profile:
        print(f"\n🎵 {profile['genre'].upper()}")
        print(f"   Mood: {profile['mood']}")
        print(f"   Popularity: {profile['avg_popularity']} | Energy: {profile['avg_energy']} | Valence: {profile['avg_valence']}")
        print(f"   Top artists: {profile['top_artists'][:3]}")


🎵 POP
   Mood: Balanced
   Popularity: 47.9 | Energy: 0.606 | Valence: 0.507
   Top artists: ['Sam Smith;Kim Petras', 'David Guetta;Bebe Rexha', 'Chris Brown']

🎵 JAZZ
   Mood: Balanced
   Popularity: 13.6 | Energy: 0.353 | Valence: 0.49
   Top artists: ['Earth, Wind & Fire', 'Grover Washington, Jr.;Bill Withers', 'Norah Jones']

🎵 METAL
   Mood: Balanced
   Popularity: 43.7 | Energy: 0.841 | Valence: 0.418
   Top artists: ['Ghost', 'Linkin Park', 'Bon Jovi']

🎵 K-POP
   Mood: Balanced
   Popularity: 57.0 | Energy: 0.676 | Valence: 0.557
   Top artists: ['IVE', 'BLACKPINK', 'TWICE']

🎵 SAD
   Mood: Balanced
   Popularity: 52.4 | Energy: 0.462 | Valence: 0.422
   Top artists: ['Powfu;beabadoobee', 'Cheriimoya;Sierra Kidd', 'Kina;Snøw']


In [5]:
def simulate_playlist_engagement(track_names, n_simulations=10000):
    """
    Monte Carlo simulation of playlist engagement.
    Models listener drop-off probability based on energy flow between tracks.
    """
    # Get track data
    playlist_tracks = []
    for name in track_names:
        mask = df['track_name'].str.lower().str.contains(name.lower())
        if mask.any():
            track = df[mask].iloc[0]
            playlist_tracks.append(track)
    
    if len(playlist_tracks) < 2:
        return None, "Need at least 2 valid tracks"
    
    n_tracks = len(playlist_tracks)
    energies     = [t['energy'] for t in playlist_tracks]
    danceability = [t['danceability'] for t in playlist_tracks]
    popularities = [t['popularity'] / 100 for t in playlist_tracks]
    
    completion_counts = np.zeros(n_tracks)
    full_completions  = 0
    
    for _ in range(n_simulations):
        listening = True
        for i in range(n_tracks):
            if not listening:
                break
            
            # Base retention from popularity
            base_retention = 0.6 + popularities[i] * 0.3
            
            # Energy flow penalty — big jumps in energy cause drop-off
            if i > 0:
                energy_jump = abs(energies[i] - energies[i-1])
                energy_penalty = energy_jump * 0.2
            else:
                energy_penalty = 0
            
            # Random listener behavior
            random_factor = np.random.normal(0, 0.05)
            
            # Retention probability
            retention = base_retention - energy_penalty + random_factor
            retention = np.clip(retention, 0, 1)
            
            if np.random.random() < retention:
                completion_counts[i] += 1
            else:
                listening = False
        
        if listening:
            full_completions += 1
    
    results = {
        'playlist_length': n_tracks,
        'simulations': n_simulations,
        'full_completion_rate': round(full_completions / n_simulations * 100, 1),
        'track_retention': [round(c / n_simulations * 100, 1) for c in completion_counts],
        'avg_tracks_heard': round(sum(completion_counts) / n_simulations, 2),
        'track_names': [t['track_name'] for t in playlist_tracks],
        'verdict': 'Strong playlist' if full_completions/n_simulations > 0.6 
                   else 'Average playlist' if full_completions/n_simulations > 0.4 
                   else 'High drop-off risk'
    }
    return results, None

# Test
playlist = ["Blinding Lights", "Shape of You", "Bad Guy", "Despacito", "Bohemian Rhapsody"]
results, error = simulate_playlist_engagement(playlist)

if results:
    print(f"🎵 Playlist Simulation ({results['simulations']:,} scenarios)")
    print(f"{'='*50}")
    print(f"Full completion rate: {results['full_completion_rate']}%")
    print(f"Avg tracks heard:     {results['avg_tracks_heard']}/{results['playlist_length']}")
    print(f"Verdict: {results['verdict']}")
    print(f"\nTrack-by-track retention:")
    for track, retention in zip(results['track_names'], results['track_retention']):
        bar = '█' * int(retention/5)
        print(f"  {track[:30]:30s} {retention:5.1f}% {bar}")

🎵 Playlist Simulation (10,000 scenarios)
Full completion rate: 9.7%
Avg tracks heard:     1.4/5
Verdict: High drop-off risk

Track-by-track retention:
  Blinding Lights                 59.9% ███████████
  Shape Of You                    34.2% ██████
  bad guy                         21.9% ████
  Despacito                       13.8% ██
  Bohemian Rhapsody - Acoustic     9.7% █


In [10]:
import pickle, os

os.makedirs('../models', exist_ok=True)

# Save updated scaler and feature matrix
with open('../models/similarity_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

np.save('../models/feature_matrix.npy', feature_matrix)

print("✅ similarity_scaler.pkl saved")
print("✅ label_encoder.pkl saved")
print("✅ feature_matrix.npy saved")

✅ similarity_scaler.pkl saved
✅ label_encoder.pkl saved
✅ feature_matrix.npy saved


In [11]:
import pickle, os

# Save scaler and feature matrix
os.makedirs('../models', exist_ok=True)
with open('../models/similarity_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

np.save('../models/feature_matrix.npy', feature_matrix)
df.to_csv('../data/spotify_clean.csv', index=False)

print("✅ similarity_scaler.pkl saved")
print("✅ feature_matrix.npy saved")
print("✅ spotify_clean.csv updated")

# Write analytics functions to src/
analytics_code = '''
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA = os.path.join(BASE, "data")
MODELS = os.path.join(BASE, "models")

def _load_data():
    return pd.read_csv(os.path.join(DATA, "spotify_clean.csv"))

def _load_similarity():
    df = _load_data()
    scaler = pickle.load(open(os.path.join(MODELS, "similarity_scaler.pkl"), "rb"))
    matrix = np.load(os.path.join(MODELS, "feature_matrix.npy"))
    return df, scaler, matrix

SIMILARITY_FEATURES = [
    "danceability","energy","loudness","speechiness",
    "acousticness","instrumentalness","liveness","valence",
    "tempo","mood_score","dance_energy"
]

def get_top_genres(n=10):
    df = _load_data()
    return df.groupby("track_genre").agg(
        avg_popularity=("popularity","mean"),
        track_count=("track_name","count"),
        avg_danceability=("danceability","mean"),
        avg_energy=("energy","mean"),
        avg_valence=("valence","mean"),
        mood_score=("mood_score","mean")
    ).reset_index().sort_values("avg_popularity", ascending=False).head(n).round(3)

def get_top_artists(n=10):
    df = _load_data()
    stats = df.groupby("artists").agg(
        avg_popularity=("popularity","mean"),
        track_count=("track_name","count"),
        avg_energy=("energy","mean"),
        avg_danceability=("danceability","mean")
    ).reset_index()
    return stats[stats["track_count"] >= 2].sort_values("avg_popularity", ascending=False).head(n).round(3)

def find_similar_tracks(track_name, n=5):
    df, scaler, matrix = _load_similarity()
    mask = df["track_name"].str.lower().str.contains(track_name.lower())
    if not mask.any():
        return None
    idx = df[mask].index[0]
    sims = cosine_similarity(matrix[idx].reshape(1,-1), matrix)[0]
    top_idx = sims.argsort()[::-1][1:n+1]
    results = df.iloc[top_idx][["track_name","artists","track_genre","popularity"]].copy()
    results["similarity"] = sims[top_idx].round(3)
    return results

def get_genre_mood_profile(genre):
    df = _load_data()
    g = df[df["track_genre"].str.lower().str.contains(genre.lower())]
    if len(g) == 0:
        return None
    v = g["valence"].mean()
    e = g["energy"].mean()
    if v > 0.6 and e > 0.6: mood = "Energetic & Happy"
    elif v > 0.6: mood = "Calm & Positive"
    elif e > 0.6: mood = "Intense & Dark"
    elif v <= 0.4 and e <= 0.4: mood = "Melancholic & Quiet"
    else: mood = "Balanced"
    return {
        "genre": genre, "mood": mood,
        "avg_popularity": round(g["popularity"].mean(),1),
        "avg_energy": round(e,3),
        "avg_valence": round(v,3),
        "avg_danceability": round(g["danceability"].mean(),3),
        "track_count": len(g),
        "top_artists": g.nlargest(3,"popularity")["artists"].tolist()
    }

def simulate_playlist(track_names, n_simulations=10000):
    df = _load_data()
    tracks = []
    for name in track_names:
        mask = df["track_name"].str.lower().str.contains(name.lower())
        if mask.any():
            tracks.append(df[mask].iloc[0])
    if len(tracks) < 2:
        return None
    energies = [t["energy"] for t in tracks]
    pops = [t["popularity"]/100 for t in tracks]
    n = len(tracks)
    counts = np.zeros(n)
    full = 0
    for _ in range(n_simulations):
        active = True
        for i in range(n):
            if not active: break
            ret = 0.6 + pops[i]*0.3 - (abs(energies[i]-energies[i-1])*0.2 if i>0 else 0) + np.random.normal(0,0.05)
            ret = np.clip(ret,0,1)
            if np.random.random() < ret: counts[i] += 1
            else: active = False
        if active: full += 1
    rate = full/n_simulations
    return {
        "full_completion_rate": round(rate*100,1),
        "track_retention": [round(c/n_simulations*100,1) for c in counts],
        "avg_tracks_heard": round(sum(counts)/n_simulations,2),
        "playlist_length": n,
        "track_names": [t["track_name"] for t in tracks],
        "verdict": "Strong playlist" if rate>0.6 else "Average playlist" if rate>0.4 else "High drop-off risk"
    }
'''

with open('../src/analytics.py', 'w') as f:
    f.write(analytics_code)

print("✅ src/analytics.py created!")

✅ similarity_scaler.pkl saved
✅ feature_matrix.npy saved
✅ spotify_clean.csv updated
✅ src/analytics.py created!
